## 7 Прогноз для новых заявок и итоговые выводы

**Входы**

- Обученная модель и новые заявки без известного исхода

**Выходы**

- Оценки риска и классы заявок
- Итоговые выводы и условия дальнейшего применения

**Что делаем**

Покажем, как получить прогноз для трёх придуманных заявок. Они нужны для примера; фактических исходов для них нет

#### Как получить прогноз

Передаём таблицу с теми же 11 признаками, но без `loan_status`. Для каждой строки получаем оценку риска и класс по выбранному порогу\
Пропуски обрабатываются так же, как при обучении. Неизвестный грейд вне A–G или неверный набор полей вызовет ошибку. Недопустимый стаж заменяется пропуском. Новая номинальная категория не остановит расчёт: её столбцы one-hot будут нулевыми, но такой случай стоит проверить отдельно

Оценку риска не трактуем как точную индивидуальную вероятность дефолта

In [ ]:
# include: _code/prepare_applications.py

In [ ]:
def predict_applications(applications, pipeline, threshold):
    prepared = prepare_applications(applications)
    scores = pipeline.predict_proba(prepared)[:, 1]
    return pd.DataFrame({
        'Оценка риска дефолта': scores,
        'Класс при выбранном пороге': (scores >= threshold).astype(int),
    }, index=applications.index)


new_applications = pd.DataFrame([
    [28, 65000, 4, 9000, 10.8, .14, 6, 'RENT', 'EDUCATION', 'N', 'B'],
    [42, 38000, 8, 16000, 17.5, .42, 12, 'MORTGAGE', 'DEBTCONSOLIDATION', 'Y', 'E'],
    [31, 92000, np.nan, 5000, np.nan, .05, 8, 'OWN', 'PERSONAL', 'N', 'A'],
], columns=FEATURES, index=['Заявка 1', 'Заявка 2', 'Заявка 3'])
example_predictions = predict_applications(new_applications, selected_pipeline, selected_threshold)
display(new_applications)
display(example_predictions.round(4))


#### За чем следить при дальнейшем использовании

| Что проверять | Зачем |
| --- | --- |
| Поля, единицы измерения, пропуски и новые категории | Чтобы входные данные не меняли смысл |
| Изменение распределений по сравнению с train | Чтобы заметить, что новые заявки стали другими |
| Качество после появления фактических исходов | Чтобы понять, сохраняются ли точность, полнота и калибровка |
| Стоимость ошибок и качество для разных групп | Чтобы оценить последствия применения модели |
| Версии данных, обработки и порога | Чтобы знать, какой именно вариант дал прогноз |

Ниже сравним доли пропусков в train и трёх заявках. На практике такую проверку проводят на потоке новых данных; по трём строкам нельзя оценить изменение распределений или качество модели

In [ ]:
monitoring_reference = pd.DataFrame({
    'Доля пропусков train': X_train.isna().mean(),
    'Доля пропусков примера': new_applications.isna().mean(),
})
display(monitoring_reference.round(4))


### Итоговая таблица выводов

In [ ]:
# include: _code/final_conclusions.py